# Leakage-aware entity sentiment analysis

This notebook is the visual companion to the tested `sentiment_analysis` package. It profiles the data, demonstrates why a row-random split is unsafe, verifies the grouped boundary, and presents the checked-in experiment. Training lives in the CLI so notebook execution order cannot silently change the methodology.

## Setup

From the repository root, install the project with `python -m pip install -e ".[notebook]"`. The path logic below works whether Jupyter starts in the repository root or in `notebooks/`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sentiment_analysis.data import (
    LABELS,
    load_dataset,
    split_diagnostics,
    stratified_group_holdout,
)

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter in the repository or notebooks directory"
sns.set_theme(style="whitegrid", context="notebook")

## Validated data profile

The loader assigns the four documented columns, rejects unknown labels, trims text, removes unusable rows and exact duplicate rows, and fingerprints the input. Similar augmented variants remain in the data and are handled by grouping—not silently discarded.

In [ ]:
training, profile = load_dataset(ROOT / "data/twitter_training.csv")
pd.Series(profile.to_dict(), name="value")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(
    data=training,
    x="sentiment",
    order=LABELS,
    hue="sentiment",
    palette="Set2",
    legend=False,
    ax=axes[0],
)
axes[0].set(title="Usable records by sentiment", xlabel="", ylabel="Rows")
top_entities = training["entity"].value_counts().head(10).sort_values()
top_entities.plot.barh(ax=axes[1], color="#4C78A8")
axes[1].set(title="Ten most frequent entities", xlabel="Rows", ylabel="")
fig.tight_layout()

## Why the supplied validation file is not an independent holdout

The dataset is built around augmented families. The supplied validation records reuse training IDs, and many reuse exact training text. Merging both files—as the legacy notebook did—makes the problem worse.

In [ ]:
validation, _ = load_dataset(ROOT / "data/twitter_validation.csv")
pd.Series(
    {
        "validation_rows": len(validation),
        "shared_tweet_ids": len(set(training["tweet_id"]) & set(validation["tweet_id"])),
        "shared_exact_texts": len(set(training["text"]) & set(validation["text"])),
    },
    name="overlap",
)

## Audited evaluation boundary

The package builds connected components over shared tweet IDs **and** exact text hashes. Each component stays entirely within one fold. This is stricter than grouping by ID alone because exact text can occur under different IDs.

In [ ]:
development, holdout = stratified_group_holdout(training, n_splits=5, random_state=42)
pd.Series(split_diagnostics(development, holdout), name="value")

## Verified experiment

`reports/experiment.json` is generated by the same train command that persists the full preprocessing-and-model artifact. The holdout is evaluated once after grouped cross-validation chooses the regularization strength.

In [ ]:
evidence = json.loads((ROOT / "reports/experiment.json").read_text(encoding="utf-8"))
pd.Series(
    {
        "grouped_cv_macro_f1": evidence["selection"]["best_cv_macro_f1"],
        "holdout_accuracy": evidence["holdout"]["accuracy"],
        "holdout_macro_precision": evidence["holdout"]["macro_precision"],
        "holdout_macro_recall": evidence["holdout"]["macro_recall"],
        "holdout_macro_f1": evidence["holdout"]["macro_f1"],
    },
    name="score",
)

In [ ]:
labels = evidence["holdout"]["label_order"]
confusion = pd.DataFrame(evidence["holdout"]["confusion_matrix"], index=labels, columns=labels)
per_class = pd.DataFrame(evidence["holdout"]["classification_report"]).T.loc[labels]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set(title="Untouched-holdout confusion matrix", xlabel="Predicted", ylabel="Actual")
per_class["f1-score"].sort_values().plot.barh(ax=axes[1], color="#F58518", xlim=(0, 1))
axes[1].set(title="Per-class holdout F1", xlabel="F1", ylabel="")
fig.tight_layout()

## Interpretation

The 53.96% macro-F1 is lower than the legacy notebook's 84.9% random-split score, but it answers the more credible question: how well does the model generalize to unseen duplicate families? Negative sentiment is strongest; `Irrelevant` remains the clearest improvement opportunity.

Next experiments should preserve the same connected-group boundary while testing entity-aware features, calibrated transformer baselines, semantic-near-duplicate detection, and error slices by entity. The holdout must remain untouched until a final comparison.